In [17]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(81426, 31)


In [18]:
pip install mlxtend

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
baskets = (
    df.groupby("orderNumber")
    ["skuNumber"]
    .apply(list)
    .tolist()
)

len(baskets)

30131

In [20]:
from mlxtend.preprocessing import (
    TransactionEncoder
)

te = TransactionEncoder()

basket_matrix = te.fit(
    baskets
).transform(
    baskets
)

basket_df = pd.DataFrame(
    basket_matrix,
    columns=te.columns_
)

basket_df.shape

(30131, 197)

In [21]:
from mlxtend.frequent_patterns import (
    fpgrowth
)

frequent_items = fpgrowth(
    basket_df,
    min_support=0.01,
    use_colnames=True
)

frequent_items.head()

,support,itemsets
0,0.144967,frozenset({SKUPM4})
1,0.066476,frozenset({SKU00033})
2,0.433706,frozenset({SKU00086})
3,0.174737,frozenset({SKU00084})
4,0.438950,frozenset({SKU00060})


In [22]:
from mlxtend.frequent_patterns import (
    association_rules
)

rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.2
)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({SKUPM4}),frozenset({SKU00084}),0.144967,0.174737,0.043278,0.298535,1.708481,1.0,0.017947,1.176485,0.484993,0.156561,0.150010,0.273104
1,frozenset({SKU00084}),frozenset({SKUPM4}),0.174737,0.144967,0.043278,0.247673,1.708481,1.0,0.017947,1.136518,0.502488,0.156561,0.120120,0.273104
2,frozenset({SKUPM4}),frozenset({SKU00060}),0.144967,0.438950,0.035943,0.247940,0.564847,1.0,-0.027690,0.746017,-0.473963,0.065593,-0.340452,0.164912
3,frozenset({SKUPM4}),frozenset({SKU00086}),0.144967,0.433706,0.035678,0.246108,0.567453,1.0,-0.027196,0.751161,-0.471318,0.065705,-0.331273,0.164185
4,"frozenset({SKUPM4, SKU00084})",frozenset({SKU00060}),0.043278,0.438950,0.015167,0.350460,0.798406,1.0,-0.003830,0.863765,-0.208809,0.032474,-0.157722,0.192507


In [23]:
rules = (
    rules.sort_values(
        "lift",
        ascending=False
    )
)

rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

,antecedents,consequents,support,confidence,lift
104,"frozenset({SKU101, SKU00060})","frozenset({SKU00086, SKU00084})",0.025588,0.702826,10.041179
107,"frozenset({SKU00086, SKU00084})","frozenset({SKU101, SKU00060})",0.025588,0.365576,10.041179
105,"frozenset({SKU101, SKU00086})","frozenset({SKU00084, SKU00060})",0.025588,0.707339,9.945331
106,"frozenset({SKU00084, SKU00060})","frozenset({SKU101, SKU00086})",0.025588,0.359776,9.945331
34,"frozenset({SKU00086, SKU00033})","frozenset({SKU00084, SKU00060})",0.013176,0.577875,8.125033
77,frozenset({SKU613}),frozenset({SKU612}),0.015831,0.328512,8.106803
78,frozenset({SKU612}),frozenset({SKU613}),0.015831,0.390663,8.106803
36,"frozenset({SKU00060, SKU00033})","frozenset({SKU00086, SKU00084})",0.013176,0.563920,8.056656
102,"frozenset({SKU00086, SKU00084, SKU00060})",frozenset({SKU101}),0.025588,0.370851,5.862604
108,frozenset({SKU101}),"frozenset({SKU00086, SKU00084, SKU00060})",0.025588,0.404512,5.862604


In [24]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName"
        ]
    ]
    .drop_duplicates()
)

sku_dict = dict(
    zip(
        sku_lookup["skuNumber"],
        sku_lookup["itemName"]
    )
)

In [25]:
rules["antecedent_names"] = (
    rules["antecedents"]
    .apply(
        lambda x:
        [
            sku_dict.get(i, i)
            for i in x
        ]
    )
)

rules["consequent_names"] = (
    rules["consequents"]
    .apply(
        lambda x:
        [
            sku_dict.get(i, i)
            for i in x
        ]
    )
)

In [26]:
rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent_names,consequent_names,confidence,lift
104,"[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]",0.702826,10.041179
107,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]","[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]",0.365576,10.041179
105,"[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]",0.707339,9.945331
106,"[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]","[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...",0.359776,9.945331
34,"[TZ 00 (with Silver) 0.45g, Rajnigandha 100g ]","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]",0.577875,8.125033
77,[Center Fresh],[Center Fruit],0.328512,8.106803
78,[Center Fruit],[Center Fresh],0.390663,8.106803
36,"[Rajnigandha 4g , Rajnigandha 100g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]",0.563920,8.056656
102,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs...",[Rajnigandha 17g Zipper IC ],0.370851,5.862604
108,[Rajnigandha 17g Zipper IC ],"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs...",0.404512,5.862604


In [27]:
print(rules.shape)

rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

(125, 16)


,antecedent_names,consequent_names,confidence,lift
104,"[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]",0.702826,10.041179
107,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]","[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]",0.365576,10.041179
105,"[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]",0.707339,9.945331
106,"[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]","[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...",0.359776,9.945331
34,"[TZ 00 (with Silver) 0.45g, Rajnigandha 100g ]","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]",0.577875,8.125033
77,[Center Fresh],[Center Fruit],0.328512,8.106803
78,[Center Fruit],[Center Fresh],0.390663,8.106803
36,"[Rajnigandha 4g , Rajnigandha 100g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]",0.563920,8.056656
102,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs...",[Rajnigandha 17g Zipper IC ],0.370851,5.862604
108,[Rajnigandha 17g Zipper IC ],"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs...",0.404512,5.862604


In [28]:
rules_save["antecedents"] = (
    rules_save["antecedents"]
    .astype(str)
)

rules_save["consequents"] = (
    rules_save["consequents"]
    .astype(str)
)

rules_save["antecedent_names"] = (
    rules_save["antecedent_names"]
    .astype(str)
)

rules_save["consequent_names"] = (
    rules_save["consequent_names"]
    .astype(str)
)

In [29]:
rules_save = rules.copy()

In [30]:
rules_save["antecedents"] = (
    rules_save["antecedents"]
    .astype(str)
)

rules_save["consequents"] = (
    rules_save["consequents"]
    .astype(str)
)

rules_save["antecedent_names"] = (
    rules_save["antecedent_names"]
    .astype(str)
)

rules_save["consequent_names"] = (
    rules_save["consequent_names"]
    .astype(str)
)

In [31]:
rules_save.to_parquet(
    "../data/processed/association_rules.parquet",
    index=False
)

print("Association rules saved")

Association rules saved


In [32]:
rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent_names,consequent_names,confidence,lift
104,"[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]",0.702826,10.041179
107,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]","[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]",0.365576,10.041179
105,"[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]",0.707339,9.945331
106,"[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]","[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...",0.359776,9.945331
34,"[TZ 00 (with Silver) 0.45g, Rajnigandha 100g ]","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]",0.577875,8.125033
77,[Center Fresh],[Center Fruit],0.328512,8.106803
78,[Center Fruit],[Center Fresh],0.390663,8.106803
36,"[Rajnigandha 4g , Rajnigandha 100g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]",0.563920,8.056656
102,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs...",[Rajnigandha 17g Zipper IC ],0.370851,5.862604
108,[Rajnigandha 17g Zipper IC ],"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs...",0.404512,5.862604


In [33]:
rules_save = rules.copy()

for col in [
    "antecedents",
    "consequents",
    "antecedent_names",
    "consequent_names"
]:
    rules_save[col] = rules_save[col].astype(str)

rules_save.to_csv(
    "../data/processed/association_rules.csv",
    index=False
)

In [34]:
print(rules.shape)

(125, 16)


In [35]:
rules.shape

(125, 16)

In [36]:
rules.shape

(125, 16)

In [37]:
rules_save = rules.copy()

In [38]:
rules_save["antecedents"] = (
    rules_save["antecedents"]
    .astype(str)
)

rules_save["consequents"] = (
    rules_save["consequents"]
    .astype(str)
)

In [39]:
if "antecedent_names" in rules_save.columns:
    rules_save["antecedent_names"] = (
        rules_save["antecedent_names"]
        .astype(str)
    )

if "consequent_names" in rules_save.columns:
    rules_save["consequent_names"] = (
        rules_save["consequent_names"]
        .astype(str)
    )

In [40]:
rules_save.to_parquet(
    "../data/processed/association_rules.parquet",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [41]:
print(rules.shape)

(125, 16)


In [42]:
rules_save = rules.copy()

In [43]:
rules_save["antecedents"] = (
    rules_save["antecedents"]
    .astype(str)
)

rules_save["consequents"] = (
    rules_save["consequents"]
    .astype(str)
)

rules_save["antecedent_names"] = (
    rules_save["antecedent_names"]
    .astype(str)
)

rules_save["consequent_names"] = (
    rules_save["consequent_names"]
    .astype(str)
)

In [44]:
rules_save.to_parquet(
    "../data/processed/association_rules.parquet",
    index=False
)

print("Saved successfully")

Saved successfully


In [45]:
print(rules.shape)
rules.head()

(125, 16)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_names,consequent_names
104,"frozenset({SKU101, SKU00060})","frozenset({SKU00086, SKU00084})",0.036408,0.069994,0.025588,0.702826,10.041179,1.0,0.023040,3.129498,0.934431,0.316632,0.680460,0.534201,"[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]","[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]"
107,"frozenset({SKU00086, SKU00084})","frozenset({SKU101, SKU00060})",0.069994,0.036408,0.025588,0.365576,10.041179,1.0,0.023040,1.518846,0.968177,0.316632,0.341605,0.534201,"[TZ 00 (with Silver) 0.45g, TZ 00 4.25g (6 Pcs)]","[Rajnigandha 17g Zipper IC , Rajnigandha 4g ]"
105,"frozenset({SKU101, SKU00086})","frozenset({SKU00084, SKU00060})",0.036175,0.071123,0.025588,0.707339,9.945331,1.0,0.023015,3.173907,0.933210,0.313160,0.684931,0.533558,"[Rajnigandha 17g Zipper IC , TZ 00 (with Silve...","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]"
106,"frozenset({SKU00084, SKU00060})","frozenset({SKU101, SKU00086})",0.071123,0.036175,0.025588,0.359776,9.945331,1.0,0.023015,1.505449,0.968320,0.313160,0.335746,0.533558,"[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]","[Rajnigandha 17g Zipper IC , TZ 00 (with Silve..."
34,"frozenset({SKU00086, SKU00033})","frozenset({SKU00084, SKU00060})",0.022800,0.071123,0.013176,0.577875,8.125033,1.0,0.011554,2.200478,0.897384,0.163173,0.545553,0.381565,"[TZ 00 (with Silver) 0.45g, Rajnigandha 100g ]","[TZ 00 4.25g (6 Pcs), Rajnigandha 4g ]"
